# Approach 2 Training Notebook (3D Geometry + MobileFaceNet)

Notebook ini memakai pipeline hybrid:
1. 3D geometry features dari MediaPipe landmarks
2. 2D image embedding dari MobileFaceNet-style backbone
3. Feature fusion untuk klasifikasi 5 kelas bentuk wajah

In [ ]:
from __future__ import annotations

import csv
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(tf.__version__)

In [ ]:
@dataclass
class TrainingConfig:
    image_size: Tuple[int, int] = (160, 160)
    batch_size: int = 32
    epochs: int = 20
    learning_rate: float = 1e-3
    val_ratio: float = 0.15
    class_csv_filename: str = "landmarks.csv"
    embedding_dim: int = 128

    @staticmethod
    def _detect_project_root() -> Path:
        cwd = Path.cwd().resolve()
        if (cwd / "output" / "training_set").exists():
            return cwd
        if (cwd.parent / "output" / "training_set").exists():
            return cwd.parent
        raise FileNotFoundError("Cannot find project root containing output/training_set")

    def __post_init__(self):
        self.project_root = self._detect_project_root()
        self.output_root = self.project_root / "output"
        self.checkpoint_dir = self.project_root / "training_notebook" / "checkpoints"
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)


config = TrainingConfig()
print("project_root:", config.project_root)
print("output_root:", config.output_root)

In [ ]:
class GeometryFeatureExtractor:
    """Extract compact geometry descriptors from flattened 3D landmarks."""

    # MediaPipe Face Mesh landmark indices (subset)
    LEFT_CHEEK = 234
    RIGHT_CHEEK = 454
    LEFT_JAW = 172
    RIGHT_JAW = 397
    CHIN = 152
    FOREHEAD = 10
    NOSE_TIP = 1

    @staticmethod
    def _safe_point(points: np.ndarray, idx: int) -> np.ndarray:
        if idx >= len(points):
            return np.zeros(3, dtype=np.float32)
        return points[idx]

    @staticmethod
    def _dist(a: np.ndarray, b: np.ndarray) -> float:
        return float(np.linalg.norm(a - b))

    @staticmethod
    def _angle(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
        ba = a - b
        bc = c - b
        denom = (np.linalg.norm(ba) * np.linalg.norm(bc)) + 1e-8
        cosine = float(np.clip(np.dot(ba, bc) / denom, -1.0, 1.0))
        return float(np.degrees(np.arccos(cosine)))

    def extract(self, flat_landmarks: np.ndarray) -> np.ndarray:
        points = flat_landmarks.reshape(-1, 3).astype(np.float32)

        left_cheek = self._safe_point(points, self.LEFT_CHEEK)
        right_cheek = self._safe_point(points, self.RIGHT_CHEEK)
        left_jaw = self._safe_point(points, self.LEFT_JAW)
        right_jaw = self._safe_point(points, self.RIGHT_JAW)
        chin = self._safe_point(points, self.CHIN)
        forehead = self._safe_point(points, self.FOREHEAD)
        nose = self._safe_point(points, self.NOSE_TIP)

        face_width = self._dist(left_cheek, right_cheek)
        face_height = self._dist(forehead, chin) + 1e-8
        jaw_width = self._dist(left_jaw, right_jaw)
        jaw_to_face_ratio = jaw_width / face_width if face_width > 0 else 0.0
        width_to_height_ratio = face_width / face_height
        chin_angle = self._angle(left_jaw, chin, right_jaw)
        depth_asymmetry = abs(float(left_cheek[2] - right_cheek[2]))
        chin_depth_offset = float(chin[2] - nose[2])

        features = np.array([
            face_width,
            face_height,
            jaw_width,
            jaw_to_face_ratio,
            width_to_height_ratio,
            chin_angle,
            depth_asymmetry,
            chin_depth_offset,
        ], dtype=np.float32)

        return features

In [ ]:
class FaceShapeDatasetBuilder:
    """Load CSV outputs and build tf.data pipelines for hybrid training."""

    def __init__(self, config: TrainingConfig, geom_extractor: GeometryFeatureExtractor):
        self.config = config
        self.geom_extractor = geom_extractor
        self.label_to_index: Dict[str, int] = {}
        self.index_to_label: Dict[int, str] = {}
        self.geom_feature_dim: int = 0

    @staticmethod
    def _extract_landmark_vector(row: Dict[str, str]) -> np.ndarray:
        values: List[float] = []
        i = 0
        while f"x_{i}" in row and f"y_{i}" in row and f"z_{i}" in row:
            values.append(float(row[f"x_{i}"]))
            values.append(float(row[f"y_{i}"]))
            values.append(float(row[f"z_{i}"]))
            i += 1
        return np.asarray(values, dtype=np.float32)

    def _resolve_image_path(self, split_name: str, class_name: str, image_name: str, image_path_raw: str) -> Path:
        raw = Path(image_path_raw)
        if raw.is_absolute() and raw.exists():
            return raw

        from_project = self.config.project_root / raw
        if from_project.exists():
            return from_project

        fallback = self.config.output_root / split_name / class_name / image_name
        return fallback

    def _scan_classes(self, split_name: str) -> List[str]:
        split_path = self.config.output_root / split_name
        if not split_path.exists():
            return []
        return sorted([p.name for p in split_path.iterdir() if p.is_dir()])

    def _read_split_records(self, split_name: str) -> List[Dict[str, object]]:
        records: List[Dict[str, object]] = []
        classes = self._scan_classes(split_name)

        for class_name in classes:
            class_csv = self.config.output_root / split_name / class_name / self.config.class_csv_filename
            if not class_csv.exists():
                continue

            with class_csv.open("r", encoding="utf-8", newline="") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    image_name = row["image_name"]
                    image_path = self._resolve_image_path(split_name, class_name, image_name, row["image_path"])
                    if not image_path.exists():
                        continue

                    landmark_vector = self._extract_landmark_vector(row)
                    if landmark_vector.size == 0:
                        continue

                    geom_features = self.geom_extractor.extract(landmark_vector)
                    self.geom_feature_dim = int(geom_features.shape[0])

                    records.append({
                        "image_path": str(image_path),
                        "class_name": class_name,
                        "geom_features": geom_features,
                    })

        return records

    def _build_label_mapping(self, class_names: List[str]) -> None:
        self.label_to_index = {name: i for i, name in enumerate(sorted(class_names))}
        self.index_to_label = {i: name for name, i in self.label_to_index.items()}

    def _split_train_validation(self, train_records: List[Dict[str, object]]) -> Tuple[List[Dict[str, object]], List[Dict[str, object]]]:
        by_class: Dict[str, List[Dict[str, object]]] = {}
        for rec in train_records:
            by_class.setdefault(rec["class_name"], []).append(rec)

        rng = np.random.default_rng(SEED)
        final_train: List[Dict[str, object]] = []
        final_val: List[Dict[str, object]] = []

        for class_name, items in by_class.items():
            idx = np.arange(len(items))
            rng.shuffle(idx)
            n_val = int(len(items) * self.config.val_ratio)
            if len(items) > 1 and n_val == 0:
                n_val = 1

            val_idx = set(idx[:n_val].tolist())
            for i, rec in enumerate(items):
                if i in val_idx:
                    final_val.append(rec)
                else:
                    final_train.append(rec)

        return final_train, final_val

    def prepare_records(self) -> Tuple[List[Dict[str, object]], List[Dict[str, object]], List[Dict[str, object]]]:
        train_all = self._read_split_records("training_set")
        test_records = self._read_split_records("testing_set")

        all_classes = sorted(set([r["class_name"] for r in train_all] + [r["class_name"] for r in test_records]))
        if not all_classes:
            raise RuntimeError("No records found in output/training_set or output/testing_set")

        self._build_label_mapping(all_classes)

        for rec in train_all:
            rec["label"] = self.label_to_index[rec["class_name"]]
        for rec in test_records:
            rec["label"] = self.label_to_index[rec["class_name"]]

        train_records, val_records = self._split_train_validation(train_all)
        return train_records, val_records, test_records

    def to_tf_dataset(self, records: List[Dict[str, object]], training: bool = False) -> tf.data.Dataset:
        image_paths = np.asarray([r["image_path"] for r in records], dtype=object)
        geom_features = np.asarray([r["geom_features"] for r in records], dtype=np.float32)
        labels = np.asarray([r["label"] for r in records], dtype=np.int32)

        ds = tf.data.Dataset.from_tensor_slices((image_paths, geom_features, labels))

        if training:
            ds = ds.shuffle(buffer_size=max(1, len(records)), seed=SEED, reshuffle_each_iteration=True)

        def _map_fn(path, geom, label):
            image_bytes = tf.io.read_file(path)
            image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
            image.set_shape([None, None, 3])
            image = tf.image.resize(image, self.config.image_size)
            image = tf.cast(image, tf.float32) / 255.0
            if training:
                image = tf.image.random_flip_left_right(image)
            return {"image_input": image, "geom_input": geom}, label

        ds = ds.map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(self.config.batch_size).prefetch(tf.data.AUTOTUNE)
        return ds

In [ ]:
class MobileFaceNetBackbone:
    """MobileFaceNet-style lightweight CNN backbone for 2D branch."""

    def __init__(self, embedding_dim: int = 128):
        self.embedding_dim = embedding_dim

    @staticmethod
    def _conv_bn_relu(x, filters: int, kernel: int, stride: int):
        x = layers.Conv2D(filters, kernel, strides=stride, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU(max_value=6.0)(x)
        return x

    @staticmethod
    def _ds_block(x, filters: int, stride: int):
        x = layers.DepthwiseConv2D(3, strides=stride, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU(max_value=6.0)(x)
        x = layers.Conv2D(filters, 1, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU(max_value=6.0)(x)
        return x

    def build(self, input_shape: Tuple[int, int, int]) -> keras.Model:
        inp = layers.Input(shape=input_shape)

        x = self._conv_bn_relu(inp, filters=64, kernel=3, stride=2)
        x = self._ds_block(x, filters=64, stride=1)
        x = self._ds_block(x, filters=128, stride=2)
        x = self._ds_block(x, filters=128, stride=1)
        x = self._ds_block(x, filters=256, stride=2)
        x = self._ds_block(x, filters=256, stride=1)

        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(self.embedding_dim, use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=1), name="image_embedding")(x)

        return keras.Model(inp, x, name="mobilefacenet_backbone")

In [ ]:
class HybridFaceShapeModel:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def build(self, num_classes: int, geom_feature_dim: int) -> keras.Model:
        image_input = layers.Input(shape=(*self.config.image_size, 3), name="image_input")
        geom_input = layers.Input(shape=(geom_feature_dim,), name="geom_input")

        backbone = MobileFaceNetBackbone(embedding_dim=self.config.embedding_dim).build((*self.config.image_size, 3))
        image_feat = backbone(image_input)

        geom_feat = layers.BatchNormalization()(geom_input)
        geom_feat = layers.Dense(64, activation="relu")(geom_feat)
        geom_feat = layers.Dropout(0.2)(geom_feat)

        fused = layers.Concatenate(name="feature_fusion")([image_feat, geom_feat])
        fused = layers.Dense(128, activation="relu")(fused)
        fused = layers.Dropout(0.3)(fused)

        output = layers.Dense(num_classes, activation="softmax", name="face_shape")(fused)

        return keras.Model(inputs=[image_input, geom_input], outputs=output, name="hybrid_face_shape_model")


class Trainer:
    def __init__(self, config: TrainingConfig, model: keras.Model):
        self.config = config
        self.model = model

    def compile(self):
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=self.config.learning_rate),
            loss=keras.losses.SparseCategoricalCrossentropy(),
            metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
        )

    def fit(self, train_ds: tf.data.Dataset, val_ds: tf.data.Dataset):
        ckpt_path = self.config.checkpoint_dir / "approach2_mobilefacenet.keras"
        callbacks = [
            keras.callbacks.ModelCheckpoint(str(ckpt_path), save_best_only=True, monitor="val_acc", mode="max"),
            keras.callbacks.EarlyStopping(monitor="val_acc", mode="max", patience=5, restore_best_weights=True),
        ]

        history = self.model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=self.config.epochs,
            callbacks=callbacks,
            verbose=1,
        )
        return history

    def evaluate(self, test_ds: tf.data.Dataset):
        return self.model.evaluate(test_ds, verbose=1)

In [ ]:
geom_extractor = GeometryFeatureExtractor()
dataset_builder = FaceShapeDatasetBuilder(config, geom_extractor)

train_records, val_records, test_records = dataset_builder.prepare_records()

print("Train records:", len(train_records))
print("Val records:", len(val_records))
print("Test records:", len(test_records))
print("Geom feature dim:", dataset_builder.geom_feature_dim)
print("Label mapping:", dataset_builder.label_to_index)

if len(train_records) == 0 or len(val_records) == 0 or len(test_records) == 0:
    raise RuntimeError("Insufficient records for train/val/test. Run preprocessing pipeline first.")

train_ds = dataset_builder.to_tf_dataset(train_records, training=True)
val_ds = dataset_builder.to_tf_dataset(val_records, training=False)
test_ds = dataset_builder.to_tf_dataset(test_records, training=False)

In [ ]:
hybrid_model = HybridFaceShapeModel(config).build(
    num_classes=len(dataset_builder.label_to_index),
    geom_feature_dim=dataset_builder.geom_feature_dim,
)

trainer = Trainer(config, hybrid_model)
trainer.compile()
hybrid_model.summary()

In [ ]:
history = trainer.fit(train_ds, val_ds)
test_metrics = trainer.evaluate(test_ds)

print("Test Loss:", float(test_metrics[0]))
print("Test Accuracy:", float(test_metrics[1]))

## Notes
- Jika training lambat, turunkan `image_size` atau `batch_size` di `TrainingConfig`.
- Jika data imbalance muncul, tambahkan class weights di `Trainer.compile()` / `model.fit()`.
- Untuk eksperimen lanjutan, bandingkan dengan baseline murni 2D (tanpa geom branch).